# transformer from scratch


<div align=center>
<img src="https://machinelearningmastery.com/wp-content/uploads/2021/08/attention_research_1.png" width="300" height="400">
</div>

In [9]:
import torch
import torch.nn as nn
import math

## 1. input embedding

所有的model不能直接处理word。transformer embedding层的作用是将word首先转换为它在vocabulary中的位置，然后再将其转换为维度为d_model的向量，我们称其为embedding。处于不同位置的token对应的embedding

target：输入input_id转换为embedding。

In [ ]:
class InputEmbedding(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super(InputEmbedding, self).__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(num_embeddings=vocab_size, 
                                      embedding_dim=d_model)
        
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

In [ ]:
# test 

inputembedding = InputEmbedding(512, 40)

X = [[1 ,2], [3, 4]]
X = torch.tensor(X, dtype=torch.long)

embedding =inputembedding(X)
print(embedding, embedding.shape)

tensor([[[ 2.0439e+01,  2.8399e+01,  4.2968e+01,  ...,  1.5213e+01,
           1.3610e+01, -1.9715e+00],
         [-1.6063e+01, -8.3334e+00,  4.1516e+00,  ...,  3.7097e-02,
          -4.4020e+01,  5.4323e+00]],

        [[-7.1160e+00,  1.2442e-01, -2.7443e+01,  ...,  1.3024e+01,
           2.9822e+00, -2.6272e+01],
         [ 1.2480e+01,  9.3464e+00, -2.5400e+00,  ...,  3.1715e+01,
          -1.7329e+01,  3.3900e+01]]], grad_fn=<MulBackward0>) torch.Size([2, 2, 512])


## 2. Positional encoding

注意到input embedding并没有添加位置信息，而我们希望model不仅要理解token的意思，还要理解到它的上下文。所以位置编码的作用是让model理解token所处的位置信息

target:输入embedding，对其加上位置编码

In [ ]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, max_seq_len: int, dropout: float):
        super(PositionalEncoding, self).__init__()

        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_seq_len, d_model)
        # shape:(max_seq_len, 1)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        # shape:(max_seq_len / 2, )
        # 为了数值稳定性：先在取对数进行运算，最后再取e变回来
        div_term = torch.exp(torch.arange(0, d_model, step=2, dtype=torch.float) 
                             * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # shape:(1,max_seq_len,d_model)
        # 因为我们的输入会添加一个维度batch, 所以这个是用来 广播
        pe = pe.unsqueeze(0) 
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape(batch, seq_len, d_model)
        # pe不是可学习参数

        # Attention
        # grad 是需要去学习理解的
        x = x + self.pe[:, :x.shape[1], :].requires_grad_(False)
        return self.dropout(x)
    
# 对算法的输出结果进行检查

pe = PositionalEncoding(512, 200, 0.2)
print(pe(torch.zeros((2, 3, 512))).shape)

torch.Size([2, 3, 512])


对上述算法的实现过程中运用了tensor的广播机制。用一句话来总结广播机制就是：将较小的维度广播至较大的维度
- 广播时从尾部维度(低维度)开始
- 两个tensor的维度
  - 相等
  - 其中一个tensor的维度尺寸为1
  - 其中一个tensor不存在这个维度

接下来总结一下广播机制下tensor设计的原则：
- 确保不保存重复数据
- 用unsqueeze将其中一个tensor的维度个数添加至指定维度个数

## 3. Layer Normalization

normalization的作用是归一化(旨在将数据调整到统一的尺度或分布，以提高模型的训练效果和稳定性)，而我们使用的归一化方法为layer normalization. 假设每个batch有n个item，layer normalization对每个item计算均值和方差进行归一化，为了数值稳定性(避免除0)在分母除方差时还会加上$\epsilon$. 对于transformer而言item即是每个token对应的embedding

此外，我们还将引入两个可学习的参数$\gamma$和$\beta$来让model自己调节值的大小

$$
\hat x_j = \frac{x_j-\mu_j}{\sqrt{\sigma_j^2+\epsilon}}
$$

In [11]:
x = torch.ones((1,2,3))
print(x, x.mean(dim=-1).shape, (x.mean(dim=-1, keepdim=True)).shape)

print(torch.ones(1).shape)
print(torch.ones(1))

tensor([[[1., 1., 1.],
         [1., 1., 1.]]]) torch.Size([1, 2]) torch.Size([1, 2, 1])
torch.Size([1])
tensor([1.])


In [ ]:
class LayerNormalization(nn.Module):
    def __init__(self, epsilon: float):
        super(LayerNormalization, self).__init__()
        
        self.epsilon = epsilon
        # 可学习的参数存放在nn.Parameter中
        self.gamma = nn.Parameter(torch.ones((1))) # Multiply
        self.beta = nn.Parameter(torch.zeros((1))) # Add

    def forward(self, x):
        # x (batch_size, sequence_length, d_model)
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return (x - mean) / (std + self.epsilon) * self.gamma + self.beta

In [18]:
x = torch.arange(0,10,dtype=torch.float).reshape((1,2,5))
ln = LayerNormalization(0.1)
output = ln(x)
print(output, output.shape)

tensor([[[-1.1897, -0.5948,  0.0000,  0.5948,  1.1897],
         [-1.1897, -0.5948,  0.0000,  0.5948,  1.1897]]],
       grad_fn=<AddBackward0>) torch.Size([1, 2, 5])
